In [4]:
#packages
import pandas as pd
import numpy as np
import re
from geopy.geocoders import Nominatim
import time
import datetime
import os

# Cleaning `accounts` data

In [5]:
# looading accounts dataset
accounts = pd.read_csv('../data/raw/HI-Small_accounts.csv')

accounts.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [6]:
print(accounts['Bank Name'].unique())

<StringArray>
[        'Portugal Bank #4507',             'Canada Bank #27',
                 'UK Bank #33',          'Germany Bank #4815',
 'National Bank of Harrisburg',             'Spain Bank #439',
       'Savings Bank of Omaha',             'Brazil Bank #39',
             'Mexico Bank #16',             'Russia Bank #39',
 ...
      'Switzerland Bank #1397',               'UK Bank #1117',
           'Israel Bank #1124',            'Crytpo Bank #904',
             'Japan Bank #681',            'Brazil Bank #218',
           'Greece Bank #2917',            'Brazil Bank #411',
             'Japan Bank #701',                'UK Bank #483']
Length: 20053, dtype: str


In [7]:
print(accounts['Entity Name'].unique())

<StringArray>
['Sole Proprietorship #50438',         'Corporation #33520',
         'Partnership #35397',         'Corporation #48813',
           'Corporation #889', 'Sole Proprietorship #42987',
 'Sole Proprietorship #20855',         'Partnership #36822',
         'Partnership #36511',         'Partnership #33830',
 ...
          'Corporation #6886',          'Partnership #2830',
 'Sole Proprietorship #16930',          'Corporation #8579',
 'Sole Proprietorship #33971',         'Corporation #40383',
         'Partnership #40687',          'Corporation #1374',
         'Corporation #46077',         'Partnership #49800']
Length: 166207, dtype: str


In [8]:
accounts['Entity Type'] = accounts['Entity Name'].str.extract(r'^([A-Za-z ]+?) #')

## Bank classification

With this section the goal was to use python and regex to work on the bank names in order to extrapolate their locations. <br> Unfortunately, due to this being syntetic data I couldn't get very far with a code based approach and had to get creative and manually modify some of the features and research approaches to cathegorize the banks.

In [9]:
# creating a new dataframe for mapping banks
bank_map = accounts[['Bank Name']].drop_duplicates().reset_index(drop=True)

### Location

In [10]:
# defining a function to extract the location name from the bank name string
def find_country(i):
    if re.search(r'Bank #\d+$', i):
        match = re.search(r'^([A-Za-z ]+?) Bank', i)
        if match:
            return match.group(1) 
    
    if re.search(r'Bank of', i):
        match = re.search(r'Bank of ([A-Za-z ]+?)$', i)
        if match:
            return match.group(1) 

    return 'other'

In [11]:
# defining a function to insert country location in place of cities
def city_to_country(df, city_country: dict) -> None:
    df['Country'] = df['Country'].map(lambda x: city_country.get(x, x))
    return df

In [12]:
# city lists to group location by country - manually identify cities in df
city_country = {
    "Albany": "USA", "Arkadelphia": "USA", "Augusta": "USA", "Billings": "USA",
    "Boston": "USA", "Bridgeport": "USA", "Butte": "USA", "Chicago": "USA",
    "Cleveland": "USA", "Columbus": "USA", "Dallas": "USA", "Danbury": "USA",
    "Denver": "USA", "Detroit": "USA", "Fairfield": "USA", "Fort Wayne": "USA",
    "Harrisburg": "USA", "Hartford": "USA", "Helena": "USA", "Houston": "USA",
    "Huron": "USA", "Indianapolis": "USA", "Lacrosse": "USA", "Los Angeles": "USA",
    "Laramie": "USA", "Madison": "USA", "Miami": "USA", "Milford": "USA",
    "New Orleans": "USA", "New York": "USA", "Newport": "USA", "Omaha": "USA",
    "Philadelphia": "USA", "Phoenix": "USA", "Pittsburgh": "USA", "Plattsburg": "USA",
    "Portland": "USA", "Providence": "USA", "Sacramento": "USA", "Seattle": "USA",
    "Springfield": "USA", "Tampa": "USA", "the East": "USA", "the North": "USA",
    "the South": "USA", "the Valley": "USA", "the West": "USA", "Topeka": "USA",
    "Tuscon": "USA", "Watertown": "USA",
    "Lincoln": "UK", "Newbury": "UK", "Portsmouth": "UK",
    "Montpelier": "Canada",
}

In [13]:
bank_map['Country'] = bank_map['Bank Name'].apply(find_country)
bank_map = city_to_country(bank_map, city_country)

In [14]:
# fixing some typos and capitalization
bank_map['Country'] = bank_map['Country'].replace('Crytpo', 'Crypto')

#### Coordinates

In [15]:
# extrapolating unique country names to get latitude longitude locations

countries = bank_map[['Country']].drop_duplicates().reset_index(drop=True)
countries = countries[~countries['Country'].isin(['Crypto', 'Other'])]

In [16]:
geolocator = Nominatim(user_agent="country-coords-script")

# defining a function to get coordinates
def get_coords(country):
    try:
        time.sleep(1)  # respect rate limit
        location = geolocator.geocode(country)
        if location:
            return pd.Series([location.latitude, location.longitude])
    except:
        pass
    return pd.Series([None, None])

In [17]:
countries[['Latitude', 'Longitude']] = countries['Country'].apply(get_coords)
bank_map = bank_map.merge(countries, on='Country', how='left')

### Types

In [18]:
# # defining a function to extract entity types
def find_type(name):
    n = name.lower()
    
    if 'credit union' in n or 'cooperative' in n:
        return 'Cooperative'
    elif 'bancorp' in n:
        return 'Holding'
    elif 'trust' in n:
        return 'Trust'
    elif 'savings' in n or 'thrift' in n:
        return 'Savings'
    elif 'bank' in n:
        return 'Commercial'
    else:
        return None

In [19]:
bank_map['Bank Type'] = bank_map['Bank Name'].apply(find_type)

In [20]:
accounts = accounts.merge(bank_map, on='Bank Name', how='left')

In [22]:
accounts.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name,Entity Type,Country,Latitude,Longitude,Bank Type
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438,Sole Proprietorship,Portugal,39.662165,-8.135352,Commercial
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520,Corporation,Canada,61.066692,-107.991707,Commercial
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397,Partnership,UK,54.702354,-3.276575,Commercial
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813,Corporation,Germany,51.163818,10.447831,Commercial
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889,Corporation,USA,39.783730,-100.445882,Commercial


# Cleaning `trans` data

In [23]:
# Loading the datasets
trans = pd.read_csv('../data/raw/HI-Small_Trans.csv')

trans.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [24]:
# Separating Date and time
trans['Timestamp'] = pd.to_datetime(trans['Timestamp'], format='%Y/%m/%d %H:%M')
trans['Date'] = trans['Timestamp'].dt.date
trans['Time'] = trans['Timestamp'].dt.time

# Turning them back into datetime objects
trans['Date'] = pd.to_datetime(trans['Date'])
trans['Time'] = pd.to_datetime(trans['Time'], format='%H:%M:%S')

In [26]:
# Removing negative transactions
trans = trans[trans['Amount Paid'] > 0]

In [27]:
# Creating a column for currency mismatch between receiving and payment
trans["currency_mismatch"] = (
    trans["Payment Currency"] != trans["Receiving Currency"]
).astype(int)

In [28]:
trans = trans[["Date", "Time", "From Bank", "Account", "To Bank", "Account.1", 
               "Amount Received", "Receiving Currency", "Amount Paid", "Payment Currency", 
               "currency_mismatch", "Payment Format", "Is Laundering"]]
trans.head(3)

,Date,Time,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,currency_mismatch,Payment Format,Is Laundering
0,2022-09-01,1900-01-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,0,Reinvestment,0
1,2022-09-01,1900-01-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,0,Cheque,0
2,2022-09-01,1900-01-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,0,Reinvestment,0


# Merging and saving

In [29]:
# defining a function to normalize column names
def clean_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df

In [30]:
# defining a function to add a prefix to receiver and sender information
def prefix_accounts(df, prefix, rename_to):
    df = df.copy()
    df.columns = prefix + df.columns
    df = df.rename(columns={f"{prefix}account_number": rename_to})
    return df

In [31]:
# cleaning column names
accounts = clean_columns(accounts)
trans = clean_columns(trans)

In [32]:
s_accounts = prefix_accounts(accounts, "s_", "account")
r_accounts = prefix_accounts(accounts, "r_", "account.1")

aml_clean = trans.merge(s_accounts, on="account", how="left")
aml_clean = aml_clean.merge(r_accounts, on="account.1", how="left")

In [33]:
# merging all datasets

aml_clean = trans.merge(s_accounts, on="account", how="left")
aml_clean = aml_clean.merge(r_accounts, on="account.1", how="left")

In [34]:
aml_clean.head(3)

,date,time,from_bank,account,to_bank,account.1,amount_received,receiving_currency,amount_paid,payment_currency,...,s_bank_type,r_bank_name,r_bank_id,r_entity_id,r_entity_name,r_entity_type,r_country,r_latitude,r_longitude,r_bank_type
0,2022-09-01,1900-01-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,...,Commercial,National Bank of Laramie,10,800D232D0,Partnership #1,Partnership,USA,39.783730,-100.445882,Commercial
1,2022-09-01,1900-01-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,...,Cooperative,Arbor Savings Bank,1,800AA5D20,Corporation #1,Corporation,other,18.515756,-77.834797,Savings
2,2022-09-01,1900-01-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,...,Commercial,National Bank of Fort Wayne,3209,800FBB3A0,Partnership #3,Partnership,USA,39.783730,-100.445882,Commercial


In [ ]:
output_path = '../../data/processed/aml_clean.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

if not os.path.exists(output_path):
    aml_clean.to_csv(output_path, index=False)